# 01 — Data Exploration

> **Food Delivery Analytics Project**  
> This notebook provides an initial inspection of the raw dataset:
> shape, data types, missing values, duplicates, and first statistical observations.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 100})
sns.set_theme(style='whitegrid', palette='deep')
print('Libraries loaded OK')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/raw/food_delivery_orders.csv', low_memory=False)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 2. Column Names & Data Types

In [ ]:
df.dtypes.to_frame('dtype')

## 3. Descriptive Statistics

In [ ]:
df.describe(include='all').T

## 4. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

In [ ]:
# Visualise missing values
missing_df = pd.DataFrame({'column': missing.index, 'count': missing.values, 'pct': missing_pct.values})
missing_df = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(missing_df['column'], missing_df['pct'], color='#1a3a5c')
ax.set_xlabel('Missing %')
ax.set_title('Missing Value Percentage by Column')
plt.tight_layout()
plt.show()

## 5. Duplicate Detection

In [ ]:
total_dups  = df.duplicated().sum()
order_dups  = df.duplicated(subset='order_id').sum()
print(f'Total duplicate rows : {total_dups}')
print(f'Duplicate order_ids  : {order_dups}')
df[df.duplicated(subset='order_id', keep=False)].head(4)

## 6. Categorical Value Counts

In [ ]:
cat_cols = ['order_status', 'cuisine', 'city', 'payment_method']
for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts(dropna=False).to_string())

## 7. Numeric Range Validation

In [ ]:
checks = {
    'order_amount > 0'       : (df['order_amount'] > 0).sum(),
    'final_amount >= 0'      : (df['final_amount'] >= 0).sum(),
    'customer_rating in 1-5' : df['customer_rating'].between(1, 5, inclusive='both').sum(),
    'estimated >= 0'         : (df['estimated_delivery_time'] >= 0).sum(),
    'delivery_time_mins >= 0': (df['delivery_time_minutes'].dropna() >= 0).sum(),
}
for k, v in checks.items():
    pct = v / len(df) * 100
    print(f'{k:40s}  →  {v:,} rows ({pct:.1f}%)')

## 8. Initial Observations

- The raw dataset has **~10,000+ rows** with 23 columns.
- **Duplicate `order_id`** entries exist (~0.3% of rows) — to be removed in cleaning.
- **`customer_rating`** and **`delivery_time_minutes`** are `NaN` for cancelled orders (expected).
- A small percentage of rows have missing `delivery_partner_id` and `distance_km`.
- All `order_amount` values are positive; `final_amount` accounts for discount correctly.
- Ratings are in `[1.0, 5.0]` half-star increments — no invalid values detected.
- **Next step:** `02_data_cleaning.ipynb` to clean, validate, and engineer features.